# 第五章：期权 — 花小钱买个"选择权" / Chapter 5: Options — Paying a Little for the Right to Choose

## 你将学到 / What You Will Learn
- 什么是期权（用最简单的话说）/ What an option is, in the plainest possible words
- 看涨期权(Call) vs 看跌期权(Put) / Calls vs. puts
- 买方 vs 卖方的盈亏特点 / Buyer vs. seller P&L profiles
- 怎么看期权的"收益图" / How to read an option payoff chart

---

## 5.1 用生活例子理解期权 / 5.1 An Everyday Analogy for Options

> 你看中了一套房子，现价 200万。你觉得这房子半年后会涨到 250万。
>
> You have your eye on a house currently priced at 2 M yuan. You think it will be worth 2.5 M in six months.
>
> 你和房东签了一份合同：
> **你付 5万 "定金"，获得半年后以 200万 买这套房子的权利。**
>
> You sign a contract with the owner:
> **You pay 50 000 yuan as a "deposit" and, in exchange, get the right to buy the house for 2 M in six months.**
>
> 半年后：
> - 房价涨到 250万 → 你行使权利，200万买入，赚了 250-200-5 = **45万**
> - 房价跌到 180万 → 你不买了（谁会以200万买只值180万的房？），你最多亏 **5万定金**
>
> Six months later:
> - Price at 2.5 M → exercise, buy at 2 M, pocket 2.5 − 2 − 0.05 = **0.45 M**
> - Price at 1.8 M → walk away (why pay 2 M for something worth 1.8 M?). Max loss = the **50 000 deposit**
>
> 这个"定金"就是**期权费(Premium)**。
> 这个"200万的买入权利"就是**看涨期权(Call Option)**。
> 这个"200万"就是**行权价(Strike Price, K)**。
>
> That deposit is the **premium**.
> That "right to buy at 2 M" is a **call option**.
> The 2 M is the **strike price (K)**.

## 5.2 两种期权 / 5.2 The Two Option Types

|  | 看涨期权 / Call | 看跌期权 / Put |
|--|-------------|-------------|
| 持有者有权 / Holder's right | 以行权价 **买入** / **Buy** at the strike | 以行权价 **卖出** / **Sell** at the strike |
| 什么时候赚 / When you profit | 价格 **涨** 过行权价 / Price moves **above** the strike | 价格 **跌** 破行权价 / Price moves **below** the strike |
| 到期收益 / Payoff at expiry | max(市价 - 行权价, 0) / max(Spot − K, 0) | max(行权价 - 市价, 0) / max(K − Spot, 0) |
| 生活比喻 / Everyday analogy | 买房的"定金" / A home-buyer's "deposit" | 保险（保底价卖出） / An insurance policy (floor sell price) |

## 5.3 买方 vs 卖方 / 5.3 Buyer vs. Seller

|  | 买方(Long) / Long | 卖方(Short) / Short |
|--|-----------|-------------|
| 付/收保费 / Premium flow | **付** 期权费 / **Pay** the premium | **收** 期权费 / **Receive** the premium |
| 最大亏损 / Max loss | 期权费（有限的）/ The premium (capped) | 可能很大甚至无限 / Potentially huge or unlimited |
| 最大盈利 / Max gain | 无限（Call）/ 很大（Put）/ Unlimited (call) or large (put) | 期权费（有限的） / The premium (capped) |
| 比喻 / Analogy | 买保险的人 / The policyholder | 卖保险的保险公司 / The insurance company |

> **关键理解**：买方亏损有上限（最多亏掉期权费），盈利无上限；卖方正好相反。
>
> **Key insight**: the buyer's loss is capped (the premium) while the gain is open-ended. The seller's profile is the mirror image.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, ToggleButtons
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 5.4 互动实验：期权收益图 / 5.4 Interactive Experiment: Option Payoff Chart

> 试试看：
> 1. 选择 **Call**(看涨) 还是 **Put**(看跌)
> 2. 选择 **买方** 还是 **卖方**
> 3. 调整 **行权价** 和 **期权费**
> 4. 观察左边的收益图怎么变化
>
> Try it:
> 1. Pick **Call** (bullish) or **Put** (bearish)
> 2. Pick the **long** or **short** side
> 3. Tweak the **strike** and **premium**
> 4. Watch the left-hand payoff chart reshape live
>
> 绿色区域 = 赚钱，红色区域 = 亏钱
>
> Green zone = profit, red zone = loss.

In [ ]:
def option_payoff(strike=100, premium=5.0, opt_type='Call', side='Long (buy)'):
    plt.close('all')
    prices = np.linspace(50, 150, 300)
    is_call = 'Call' in opt_type
    is_long = 'Long' in side

    if is_call:
        intrinsic = np.maximum(prices - strike, 0)
    else:
        intrinsic = np.maximum(strike - prices, 0)

    if is_long:
        pnl = intrinsic - premium
    else:
        pnl = premium - intrinsic

    if is_call:
        breakeven = strike + premium
    else:
        breakeven = strike - premium

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Left: current selection payoff
    ax = axes[0]
    ax.fill_between(prices, pnl, 0, where=(pnl >= 0), color=C_GREEN, alpha=0.3)
    ax.fill_between(prices, pnl, 0, where=(pnl < 0), color=C_RED, alpha=0.3)
    ax.plot(prices, pnl, color='black', lw=3)
    ax.axhline(y=0, color='gray', ls='--', alpha=0.5)
    ax.axvline(x=strike, color=C_PURPLE, ls='--', alpha=0.5, label=f'Strike K={strike}')
    ax.axvline(x=breakeven, color=C_ORANGE, ls=':', alpha=0.7)
    ax.annotate(f'Breakeven\n{breakeven:.0f}', xy=(breakeven, 0), xytext=(breakeven + 8, premium * 1.2),
                fontsize=10, fontweight='bold', color=C_ORANGE,
                arrowprops=dict(arrowstyle='->', color=C_ORANGE, lw=1.5))
    if is_long:
        ax.annotate(f'Max loss ${premium:.0f}', xy=(prices[0] + 10, -premium),
                    fontsize=10, color=C_RED, fontweight='bold')
    ax.set_xlabel('Spot at expiry', fontsize=11)
    ax.set_ylabel('P&L', fontsize=11)
    side_short = 'Long' if is_long else 'Short'
    type_short = 'Call' if is_call else 'Put'
    ax.set_title(f'{side_short} {type_short} Payoff', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)

    # Middle: 4 basic positions
    ax = axes[1]
    long_call = np.maximum(prices - strike, 0) - premium
    short_call = premium - np.maximum(prices - strike, 0)
    long_put = np.maximum(strike - prices, 0) - premium
    short_put = premium - np.maximum(strike - prices, 0)

    ax.plot(prices, long_call, color=C_GREEN, lw=2, label='Long Call')
    ax.plot(prices, short_call, color=C_GREEN, ls='--', lw=1.5, label='Short Call')
    ax.plot(prices, long_put, color=C_RED, lw=2, label='Long Put')
    ax.plot(prices, short_put, color=C_RED, ls='--', lw=1.5, label='Short Put')
    ax.axhline(y=0, color='gray', ls='--', alpha=0.5)
    ax.axvline(x=strike, color=C_PURPLE, ls='--', alpha=0.3)
    ax.set_xlabel('Spot at expiry', fontsize=11)
    ax.set_ylabel('P&L', fontsize=11)
    ax.set_title('Four Basic Option Positions', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.2)

    # Right: text panel
    ax = axes[2]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('Key Info', fontsize=13, fontweight='bold')

    if is_long:
        max_loss = f'${premium:.0f} (premium paid)'
        if is_call:
            max_gain = 'Unlimited (upside)'
        else:
            max_gain = f'${strike - premium:.0f} (price -> 0)'
    else:
        max_gain = f'${premium:.0f} (premium received)'
        if is_call:
            max_loss = 'Unlimited (upside)'
        else:
            max_loss = f'${strike - premium:.0f}'

    info = [
        ('Side', f'{side_short} ({"pay" if is_long else "receive"} premium)'),
        ('Type', f'{type_short} (profit when price {"rises" if is_call else "falls"})'),
        ('Strike K', f'{strike}'),
        ('Premium', f'${premium:.1f}'),
        ('Breakeven', f'{breakeven:.1f}'),
        ('Max loss', max_loss),
        ('Max gain', max_gain),
    ]
    for i, (label, val) in enumerate(info):
        y = 9 - i * 1.2
        col = C_RED if 'loss' in label else C_GREEN if 'gain' in label else 'gray'
        ax.text(0.5, y, f'{label}:', fontsize=10, fontweight='bold', color='gray')
        ax.text(5.5, y, val, fontsize=10, fontweight='bold', color=col if col != 'gray' else 'black')

    plt.tight_layout()
    plt.show()

    print(f'\n  {side_short} {type_short}:')
    if is_long:
        print(f'    You paid ${premium} for the right')
        if is_call:
            print(f'    Profit when price > {breakeven}; max loss = ${premium}')
        else:
            print(f'    Profit when price < {breakeven}; max loss = ${premium}')
    else:
        print(f'    You received ${premium} but bear assignment risk')

interact(option_payoff,
         strike=FloatSlider(value=100, min=50, max=200, step=1, description='Strike:'),
         premium=FloatSlider(value=5.0, min=0.5, max=30, step=0.5, description='Premium($):'),
         opt_type=ToggleButtons(options=['Call', 'Put'], description='Type:'),
         side=ToggleButtons(options=['Long (buy)', 'Short (sell)'], description='Side:'));

## 5.5 记住这张图 / 5.5 Memorize This Picture

```
买看涨 / Long Call              买看跌 / Long Put
     P&L ↑                          P&L ↑
         /                       \
        /                         \
───────/────── Spot →    ──────────\────── Spot →
  -premium                         -premium

"Bet on a rise; loss is capped"   "Bet on a fall; loss is capped"
```

## 5.6 小结 / 5.6 Chapter Recap

| 关键词 / Term | 一句话 / One-liner |
|--------|--------|
| 期权 / Option | 花小钱买"权利"，不是义务 / Pay a small price for a *right*, not an obligation |
| 看涨 Call / Call | 有权买入，赌涨 / Right to buy; bet on the upside |
| 看跌 Put / Put | 有权卖出，赌跌 / Right to sell; bet on the downside |
| 买方 / Buyer | 花期权费，亏损有上限 / Pays premium, loss is capped |
| 卖方 / Seller | 收期权费，但可能亏很多 / Receives premium, loss can be huge |
| 行权价 K / Strike K | 约定的买入/卖出价格 / The agreed buy / sell price |
| 盈亏平衡点 / Breakeven | Call: K+期权费，Put: K-期权费 / Call: K + premium; Put: K − premium |

> **下一章：Black-Scholes定价** —— 怎么算出一个期权"应该值多少钱"？
>
> **Next chapter: Black-Scholes pricing** — how to compute what an option *should* be worth.